# Structure Factor Calculation

Computes the 2D structure factor S(q) from a CSV of particle positions and produces 1D and 2D visualisations.

Steps:
1. Set input/output paths and calculation parameters
2. Run the structure factor calculation (GPU if available, otherwise CPU)
3. Plot the 2D S(q) map
4. Plot the 1D radial-average S(q)

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import os
from phd_tools.structure_factor import calculate_structure_factor, plot_Sq_2D, plot_Sq_1D

## 1. Paths and parameters

- `read_folder` — directory containing the input CSV file
- `filename` — CSV filename **without** the `.csv` extension
- `range_calculation` — scattering vector range in units of the mean inter-particle distance D
- `vector_step` — scattering vector step size in units of D
- `save_folder` — directory where `.dat` result files will be written

In [2]:
nb_dir      = Path.cwd()
read_folder = str((nb_dir.parent / 'data' / 'structure_files').resolve())
save_folder = str((nb_dir / 'results').resolve())
Path(save_folder).mkdir(parents=True, exist_ok=True)

filename          = 'NanoP2710N2_liftoff_08112021_06'  # change to your file
range_calculation = 10
vector_step       = 1

print('Input :', read_folder)
print('Output:', save_folder)

Input : C:\Users\Gil\Documents\GitHub\portfolio\PhD\data\structure_files
Output: C:\Users\Gil\Documents\GitHub\portfolio\PhD\notebooks\results


## 2. Calculate structure factor

The function automatically selects CUDA if a GPU is available, otherwise runs on CPU.
Results are written to `save_folder` as two `.dat` files:
- `2D_Sq_<name>_range_<R>_step_<s>.dat` — 2D S(q) with q vectors appended
- `Sq_<name>_range_<R>_step_<s>.dat` — 1D radial average

In [ ]:
calculate_structure_factor(
    read_folder=read_folder,
    file=filename,
    range_calculation=range_calculation,
    vector_step=vector_step,
    save_folder=save_folder,
    device='gpu'
)
print('Done.')

OOM at chunk=1; falling back to iterative for remaining rows


: 

## 3. Plot 2D S(q)

`edge` sets the display window as `(-edge[1], edge[1])` on both axes in the chosen `x_axis` units.
Pass `log_scale=True` for a logarithmic colour scale.

In [ ]:
# Saved file uses `range_<range_calculation>_step_<vector_step as float>` naming
result_file = f'2D_Sq_{filename}_range_{range_calculation}_step_{float(vector_step):.1f}'

plot_Sq_2D(
    folder_read=save_folder + os.sep,
    file=result_file,
    edge=(-5, 5),
    x_axis='qD',
    log_scale=False,
    max_value=50,
)

## 4. Plot 1D radial-average S(q)

`file` is a list so multiple curves can be overlaid on the same plot.
`edges` sets the x-axis range.

In [ ]:
result_1d_file = f'Sq_{filename}_range_{range_calculation}_step_{float(vector_step):.1f}'

plot_Sq_1D(
    folder_read=save_folder + os.sep,
    file=[result_1d_file],
    labels=[filename],
    edges=(0, 10),
    x_axis='qD',
    log_scale=False,
    y_max=20,
)